# Option 0D - PHE-only 2.5D Detection

Notebook này chuyển bài toán từ **segmentation** sang **detection**.

Khác các option trước:

- Không dự đoán mask PHE.
- Mask PHE chỉ được dùng để sinh **bounding box ground truth**.
- Model dự đoán **objectness/class + bbox** cho slice trung tâm.

Kiến trúc theo sơ đồ đề xuất:

```text
Input 2.5D stack N lát cắt
        |
2D CNN backbone cho từng lát
        |
2D feature maps theo từng lát
        |
3D Conv fusion trên stack feature maps
        |
FPN / Neck multi-scale
        |
Detection head kiểu FCOS/RetinaNet-lite
        |
Predict bbox + class/objectness
```

Phạm vi hiện tại:

- PHE-only.
- Không dùng Seg-CQ500.
- Không dùng INSTANCE2022.
- Không dùng teacher prior.

## 0. Dependency notes

In [ ]:
INSTALL_BASIC_DEPS = False

if INSTALL_BASIC_DEPS:
    import sys
    import subprocess
    pkgs = ["numpy", "pandas", "matplotlib", "scipy", "nibabel", "tqdm", "torchvision"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *pkgs])

In [ ]:
from pathlib import Path
from dataclasses import dataclass, asdict


@dataclass
class Option0DConfig:
    project_root: Path = Path.cwd()
    seed: int = 42

    # Run switches.
    run_training: bool = True
    run_final_eval: bool = True
    debug_small_run: bool = False
    debug_train_cases: int = 8
    debug_val_cases: int = 4
    debug_test_cases: int = 4
    debug_epochs: int = 2

    # Detection input.
    image_size: int = 512
    stack_size: int = 5
    slice_offsets: tuple = (-2, -1, 0, 1, 2)
    brain_window_low: float = -20.0
    brain_window_high: float = 100.0

    # Backbone/FPN/detection head.
    backbone: str = "resnet18"
    pretrained_backbone: bool = True
    fpn_channels: int = 128
    head_channels: int = 128
    head_depth: int = 3
    strides: tuple = (4, 8, 16, 32)
    level_names: tuple = ("p2", "p3", "p4", "p5")
    level_box_ranges: tuple = ((0, 48), (32, 96), (64, 192), (128, 100000))
    center_radius_cells: int = 1

    # Training.
    epochs: int = 80
    batch_size: int = 2
    grad_accum_steps: int = 2
    lr_backbone: float = 1e-5
    lr_head: float = 5e-5
    weight_decay: float = 1e-4
    patience: int = 16
    min_delta: float = 1e-4
    freeze_backbone_epochs: int = 2
    amp: bool = True
    num_workers: int = 0

    # Imbalance/loss.
    positive_slice_weight: float = 4.0
    focal_alpha: float = 0.25
    focal_gamma: float = 2.0
    lambda_cls: float = 1.0
    lambda_box: float = 1.5
    augment_train: bool = True

    # Inference/tuning.
    score_threshold: float = 0.20
    score_grid: tuple = (0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60)
    nms_iou: float = 0.30
    max_detections_per_slice: int = 5
    eval_iou_thresholds: tuple = (0.30, 0.50)
    save_prediction_csv: bool = True


CFG = Option0DConfig()
PROJECT_ROOT = CFG.project_root.resolve()

PHE_ROOT = PROJECT_ROOT / "PHE-SICH-CT-IDS" / "SubdatasetA_NIFIT" / "NIFIT"
PHE_IMAGE_DIR = PHE_ROOT / "set"
PHE_MASK_DIR = PHE_ROOT / "label"
OPTION1_SPLIT_PATH = PROJECT_ROOT / "outputs_option1_phe_sich_2d_unet" / "manifests" / "phe_sich_option1_patient_442_split.csv"

OUTPUT_ROOT = PROJECT_ROOT / "outputs_option_0d_phe_only_25d_detection"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIG_DIR = OUTPUT_ROOT / "figures"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
PRED_DIR = OUTPUT_ROOT / "predictions"
LOG_DIR = OUTPUT_ROOT / "logs"

for p in [OUTPUT_ROOT, MANIFEST_DIR, TABLE_DIR, FIG_DIR, CHECKPOINT_DIR, PRED_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

assert CFG.stack_size == len(CFG.slice_offsets), "stack_size must match slice_offsets length"

print("Project root:", PROJECT_ROOT)
print("PHE root    :", PHE_ROOT)
print("Output root :", OUTPUT_ROOT)
print("Backbone    :", CFG.backbone)
print("Stack       :", CFG.slice_offsets)

In [ ]:
import os
import json
import math
import random
import time
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

try:
    import nibabel as nib
    NIB_AVAILABLE = True
except Exception as exc:
    nib = None
    NIB_AVAILABLE = False
    print("nibabel unavailable:", exc)

try:
    from scipy import ndimage
    SCIPY_AVAILABLE = True
except Exception as exc:
    ndimage = None
    SCIPY_AVAILABLE = False
    print("scipy unavailable:", exc)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
    TORCH_AVAILABLE = True
except Exception as exc:
    torch = nn = F = Dataset = DataLoader = WeightedRandomSampler = None
    TORCH_AVAILABLE = False
    print("torch unavailable; training/eval cells will skip:", exc)

if TORCH_AVAILABLE:
    try:
        from torchvision.models import resnet18, ResNet18_Weights
        try:
            from torchvision.ops import nms as torchvision_nms
        except Exception:
            torchvision_nms = None
        TORCHVISION_AVAILABLE = True
    except Exception as exc:
        resnet18 = ResNet18_Weights = torchvision_nms = None
        TORCHVISION_AVAILABLE = False
        print("torchvision unavailable; detection model will skip:", exc)
else:
    resnet18 = ResNet18_Weights = torchvision_nms = None
    TORCHVISION_AVAILABLE = False


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


set_seed(CFG.seed)
DEVICE = "cuda" if TORCH_AVAILABLE and torch.cuda.is_available() else "cpu"
print({"nibabel": NIB_AVAILABLE, "scipy": SCIPY_AVAILABLE, "torch": TORCH_AVAILABLE, "torchvision": TORCHVISION_AVAILABLE, "device": DEVICE})

## 1. Data helpers and patient-level split

In [ ]:
def is_nifti(path: Path) -> bool:
    return path.name.endswith(".nii") or path.name.endswith(".nii.gz")


def strip_nii_suffix(path_or_name) -> str:
    name = Path(path_or_name).name
    if name.endswith(".nii.gz"):
        return name[:-7]
    if name.endswith(".nii"):
        return name[:-4]
    return Path(name).stem


def load_nifti(path: Path, canonical: bool = True):
    if not NIB_AVAILABLE:
        raise ImportError("Install nibabel first.")
    img = nib.load(str(path))
    if canonical:
        img = nib.as_closest_canonical(img)
    data = np.asanyarray(img.dataobj)
    spacing = tuple(float(v) for v in img.header.get_zooms()[:3])
    return data, spacing, img.affine


def header_info(path: Path) -> Dict:
    img = nib.load(str(path))
    shape = tuple(int(v) for v in img.shape[:3])
    spacing = tuple(float(v) for v in img.header.get_zooms()[:3])
    return {
        "shape": shape,
        "spacing_x": spacing[0],
        "spacing_y": spacing[1],
        "spacing_z": spacing[2],
        "n_slices": shape[2],
    }


def mask_volume_stats(mask_path: Path, spacing: Tuple[float, float, float]) -> Dict:
    mask = np.asanyarray(nib.load(str(mask_path)).dataobj) > 0
    per_slice = mask.reshape((-1, mask.shape[2])).sum(axis=0)
    voxel_ml = float(np.prod(spacing) / 1000.0)
    return {
        "mask_voxels": int(mask.sum()),
        "mask_volume_ml": float(mask.sum() * voxel_ml),
        "positive_slices": int((per_slice > 0).sum()),
    }


def build_phe_manifest() -> pd.DataFrame:
    if not PHE_IMAGE_DIR.exists() or not PHE_MASK_DIR.exists():
        raise FileNotFoundError(f"PHE folders not found: {PHE_IMAGE_DIR}, {PHE_MASK_DIR}")
    images = {strip_nii_suffix(p): p for p in PHE_IMAGE_DIR.iterdir() if p.is_file() and is_nifti(p)}
    masks = {strip_nii_suffix(p): p for p in PHE_MASK_DIR.iterdir() if p.is_file() and is_nifti(p)}
    rows = []
    for scan_id, img_path in sorted(images.items()):
        mask_path = masks.get(scan_id)
        if mask_path is None:
            continue
        info = header_info(img_path)
        spacing = (info["spacing_x"], info["spacing_y"], info["spacing_z"])
        rows.append({
            "dataset": "PHE-SICH-CT-IDS",
            "patient_id": f"phe_{scan_id}",
            "scan_id": scan_id,
            "img_path": str(img_path),
            "mask_path": str(mask_path),
            **info,
            **mask_volume_stats(mask_path, spacing),
        })
    return pd.DataFrame(rows)


def make_patient_442_split(df: pd.DataFrame, seed: int = CFG.seed) -> pd.DataFrame:
    df = df.copy().reset_index(drop=True)
    rng = np.random.default_rng(seed)
    strata = pd.qcut(df["mask_volume_ml"].rank(method="first"), q=4, labels=False)
    split = np.array([""] * len(df), dtype=object)
    for _, part in df.groupby(strata):
        idx = part.index.to_numpy().copy()
        rng.shuffle(idx)
        n = len(idx)
        n_train = int(round(n * 0.4))
        n_val = int(round(n * 0.4))
        split[idx[:n_train]] = "train"
        split[idx[n_train:n_train + n_val]] = "val"
        split[idx[n_train + n_val:]] = "test"
    df["split"] = split
    return df


if OPTION1_SPLIT_PATH.exists():
    phe_df = pd.read_csv(OPTION1_SPLIT_PATH)
    print("Loaded Option 1 split:", OPTION1_SPLIT_PATH)
else:
    phe_df = make_patient_442_split(build_phe_manifest())
    print("Built new patient-level split with seed:", CFG.seed)

if CFG.debug_small_run:
    phe_df = pd.concat([
        phe_df[phe_df["split"] == "train"].head(CFG.debug_train_cases),
        phe_df[phe_df["split"] == "val"].head(CFG.debug_val_cases),
        phe_df[phe_df["split"] == "test"].head(CFG.debug_test_cases),
    ], ignore_index=True)

split_path = MANIFEST_DIR / "option_0d_phe_split.csv"
phe_df.to_csv(split_path, index=False, encoding="utf-8")
train_rows = phe_df[phe_df["split"] == "train"].reset_index(drop=True)
val_rows = phe_df[phe_df["split"] == "val"].reset_index(drop=True)
test_rows = phe_df[phe_df["split"] == "test"].reset_index(drop=True)

display(phe_df.groupby("split").agg(
    cases=("scan_id", "count"),
    slices=("n_slices", "sum"),
    positive_slices=("positive_slices", "sum"),
    median_phe_ml=("mask_volume_ml", "median"),
    total_phe_ml=("mask_volume_ml", "sum"),
).reset_index())
print("Saved split:", split_path)

## 2. Detection preprocessing: 2.5D stack and bbox labels

In [ ]:
def brain_window(image: np.ndarray, low: float = CFG.brain_window_low, high: float = CFG.brain_window_high) -> np.ndarray:
    image = image.astype(np.float32, copy=False)
    image = np.clip(image, low, high)
    return ((image - low) / max(high - low, 1e-6)).astype(np.float32)


def resize_2d(array: np.ndarray, out_size: int, order: int = 1) -> np.ndarray:
    if array.shape[-2:] == (out_size, out_size):
        return array
    if not SCIPY_AVAILABLE:
        raise ImportError("Install scipy for resizing.")
    zoom_y = out_size / array.shape[-2]
    zoom_x = out_size / array.shape[-1]
    return ndimage.zoom(array, (zoom_y, zoom_x), order=order)


def make_25d_stack(volume: np.ndarray, z: int, image_size: int = CFG.image_size) -> np.ndarray:
    n_slices = volume.shape[2]
    channels = []
    for off in CFG.slice_offsets:
        zi = int(np.clip(z + int(off), 0, n_slices - 1))
        x = brain_window(np.asarray(volume[:, :, zi]))
        channels.append(resize_2d(x, image_size, order=1))
    return np.stack(channels, axis=0).astype(np.float32)


def resize_mask(mask_2d: np.ndarray, image_size: int = CFG.image_size) -> np.ndarray:
    return (resize_2d((mask_2d > 0).astype(np.float32), image_size, order=0) > 0.5).astype(np.uint8)


def mask_to_box(mask_2d: np.ndarray) -> Tuple[np.ndarray, bool]:
    ys, xs = np.where(mask_2d > 0)
    if len(xs) == 0:
        return np.zeros(4, dtype=np.float32), False
    x1 = float(xs.min())
    y1 = float(ys.min())
    x2 = float(xs.max() + 1)
    y2 = float(ys.max() + 1)
    x2 = max(x2, x1 + 1.0)
    y2 = max(y2, y1 + 1.0)
    return np.array([x1, y1, x2, y2], dtype=np.float32), True


def bbox_area(box: np.ndarray) -> float:
    return float(max(0.0, box[2] - box[0]) * max(0.0, box[3] - box[1]))


def largest_mask_slice(mask_volume: np.ndarray) -> int:
    per_slice = (mask_volume > 0).reshape((-1, mask_volume.shape[2])).sum(axis=0)
    return int(np.argmax(per_slice))


sample_row = train_rows.sort_values("mask_volume_ml", ascending=False).iloc[0]
image, spacing, _ = load_nifti(Path(sample_row["img_path"]))
mask, _, _ = load_nifti(Path(sample_row["mask_path"]))
z = largest_mask_slice(mask)
stack = make_25d_stack(image, z, image_size=256)
center_mask = resize_mask(mask[:, :, z], image_size=256)
box, has_box = mask_to_box(center_mask)

fig, axes = plt.subplots(1, CFG.stack_size + 1, figsize=(4 * (CFG.stack_size + 1), 4))
for i in range(CFG.stack_size):
    axes[i].imshow(stack[i], cmap="gray")
    axes[i].set_title(f"offset {CFG.slice_offsets[i]}")
    axes[i].axis("off")
axes[-1].imshow(stack[CFG.stack_size // 2], cmap="gray")
if has_box:
    x1, y1, x2, y2 = box
    axes[-1].add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2))
axes[-1].imshow(np.ma.masked_where(center_mask == 0, center_mask), cmap="autumn", alpha=0.35)
axes[-1].set_title("bbox label from mask")
axes[-1].axis("off")
plt.tight_layout()
plt.savefig(FIG_DIR / "option_0d_sample_25d_bbox.png", dpi=160, bbox_inches="tight")
plt.show()

## 3. Dataset for slice-level PHE detection

In [ ]:
if TORCH_AVAILABLE:
    class PheDetection25DDataset(Dataset):
        def __init__(self, df: pd.DataFrame, image_size: int = CFG.image_size, augment: bool = False):
            self.df = df.copy().reset_index(drop=True)
            self.image_size = int(image_size)
            self.augment = bool(augment)
            self.cache = {}
            self.slice_df = self._build_slice_index()

        def _build_slice_index(self):
            rows = []
            for row_idx, row in self.df.iterrows():
                mask, _, _ = load_nifti(Path(row["mask_path"]))
                mask = mask > 0
                n = mask.shape[2]
                per_slice = mask.reshape((-1, n)).sum(axis=0)
                for z in range(n):
                    rows.append({
                        "row_idx": row_idx,
                        "scan_id": row["scan_id"],
                        "slice_idx": int(z),
                        "positive": bool(per_slice[z] > 0),
                        "mask_pixels": int(per_slice[z]),
                    })
            return pd.DataFrame(rows)

        def _load_case(self, row_idx: int):
            if row_idx not in self.cache:
                row = self.df.iloc[row_idx]
                image, spacing, _ = load_nifti(Path(row["img_path"]))
                mask, _, _ = load_nifti(Path(row["mask_path"]))
                self.cache[row_idx] = (image.astype(np.float32), (mask > 0).astype(np.uint8), spacing)
            return self.cache[row_idx]

        def _augment(self, x: np.ndarray, y: np.ndarray, box: np.ndarray, has_box: bool):
            if random.random() < 0.5:
                x = np.flip(x, axis=2).copy()
                y = np.flip(y, axis=1).copy()
                if has_box:
                    old_x1, old_x2 = float(box[0]), float(box[2])
                    box[0] = self.image_size - old_x2
                    box[2] = self.image_size - old_x1
            if random.random() < 0.20:
                gamma = random.uniform(0.90, 1.10)
                x = np.clip(x, 0, 1) ** gamma
            if random.random() < 0.12:
                x = np.clip(x + np.random.normal(0, 0.010, size=x.shape).astype(np.float32), 0, 1)
            return x.astype(np.float32), y.astype(np.uint8), box.astype(np.float32), bool(has_box)

        def __len__(self):
            return len(self.slice_df)

        def __getitem__(self, idx):
            item = self.slice_df.iloc[idx]
            image, mask, spacing = self._load_case(int(item["row_idx"]))
            z = int(item["slice_idx"])
            x = make_25d_stack(image, z, self.image_size)
            y = resize_mask(mask[:, :, z], self.image_size)
            box, has_box = mask_to_box(y)
            if self.augment:
                x, y, box, has_box = self._augment(x, y, box, has_box)
            return {
                "image": torch.from_numpy(x).float(),
                "box": torch.tensor(box, dtype=torch.float32),
                "has_box": torch.tensor([1.0 if has_box else 0.0], dtype=torch.float32),
                "scan_id": str(item["scan_id"]),
                "slice_idx": int(z),
                "spacing": torch.tensor(spacing, dtype=torch.float32),
            }
else:
    print("Torch unavailable. Dataset skipped.")

## 4. Model: 2D backbone per slice + 3D fusion + FPN + FCOS-like head

In [ ]:
if TORCH_AVAILABLE and TORCHVISION_AVAILABLE:
    def load_resnet18_backbone():
        weights = None
        if CFG.pretrained_backbone:
            try:
                weights = ResNet18_Weights.DEFAULT
            except Exception:
                weights = None
        try:
            model = resnet18(weights=weights)
            print("ResNet18 weights:", "ImageNet" if weights is not None else "random")
        except Exception as exc:
            print("Could not load pretrained ResNet18; falling back to random:", exc)
            model = resnet18(weights=None)

        old_conv = model.conv1
        new_conv = nn.Conv2d(
            1,
            old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False,
        )
        with torch.no_grad():
            new_conv.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
        model.conv1 = new_conv
        return model


    class SliceResNet18Encoder(nn.Module):
        def __init__(self):
            super().__init__()
            base = load_resnet18_backbone()
            self.stem = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
            self.layer1 = base.layer1
            self.layer2 = base.layer2
            self.layer3 = base.layer3
            self.layer4 = base.layer4

        def forward(self, x):
            x = self.stem(x)
            c2 = self.layer1(x)
            c3 = self.layer2(c2)
            c4 = self.layer3(c3)
            c5 = self.layer4(c4)
            return {"c2": c2, "c3": c3, "c4": c4, "c5": c5}


    def gn(channels: int):
        groups = 8
        while channels % groups != 0 and groups > 1:
            groups //= 2
        return nn.GroupNorm(groups, channels)


    class Depthwise3DFusion(nn.Module):
        def __init__(self, channels: int, depth: int):
            super().__init__()
            self.depthwise = nn.Conv3d(channels, channels, kernel_size=(depth, 1, 1), groups=channels, bias=False)
            self.pointwise = nn.Sequential(
                nn.Conv2d(channels, channels, kernel_size=1, bias=False),
                gn(channels),
                nn.SiLU(inplace=True),
            )

        def forward(self, x):
            # x: [B, N, C, H, W]
            x = x.permute(0, 2, 1, 3, 4).contiguous()
            x = self.depthwise(x).squeeze(2)
            return self.pointwise(x)


    class FPN(nn.Module):
        def __init__(self, in_channels=(64, 128, 256, 512), out_channels=CFG.fpn_channels):
            super().__init__()
            self.laterals = nn.ModuleList([nn.Conv2d(c, out_channels, 1) for c in in_channels])
            self.smooth = nn.ModuleList([
                nn.Sequential(nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False), gn(out_channels), nn.SiLU(inplace=True))
                for _ in in_channels
            ])

        def forward(self, feats: Dict[str, torch.Tensor]):
            c2, c3, c4, c5 = feats["c2"], feats["c3"], feats["c4"], feats["c5"]
            lat = [l(f) for l, f in zip(self.laterals, [c2, c3, c4, c5])]
            p5 = lat[3]
            p4 = lat[2] + F.interpolate(p5, size=lat[2].shape[-2:], mode="nearest")
            p3 = lat[1] + F.interpolate(p4, size=lat[1].shape[-2:], mode="nearest")
            p2 = lat[0] + F.interpolate(p3, size=lat[0].shape[-2:], mode="nearest")
            return {
                "p2": self.smooth[0](p2),
                "p3": self.smooth[1](p3),
                "p4": self.smooth[2](p4),
                "p5": self.smooth[3](p5),
            }


    class DetectionHead(nn.Module):
        def __init__(self, channels=CFG.fpn_channels, head_channels=CFG.head_channels, depth=CFG.head_depth):
            super().__init__()
            def tower():
                layers = []
                in_ch = channels
                for _ in range(depth):
                    layers.extend([
                        nn.Conv2d(in_ch, head_channels, 3, padding=1, bias=False),
                        gn(head_channels),
                        nn.SiLU(inplace=True),
                    ])
                    in_ch = head_channels
                return nn.Sequential(*layers)

            self.cls_tower = tower()
            self.reg_tower = tower()
            self.cls_logits = nn.Conv2d(head_channels, 1, 3, padding=1)
            self.bbox_pred = nn.Conv2d(head_channels, 4, 3, padding=1)

            nn.init.constant_(self.cls_logits.bias, -4.6)

        def forward(self, features: Dict[str, torch.Tensor]):
            out = {}
            for name, x in features.items():
                out[name] = {
                    "cls": self.cls_logits(self.cls_tower(x)),
                    "box": self.bbox_pred(self.reg_tower(x)),
                }
            return out


    class Phe25DDetectionModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = SliceResNet18Encoder()
            self.fusions = nn.ModuleDict({
                "c2": Depthwise3DFusion(64, CFG.stack_size),
                "c3": Depthwise3DFusion(128, CFG.stack_size),
                "c4": Depthwise3DFusion(256, CFG.stack_size),
                "c5": Depthwise3DFusion(512, CFG.stack_size),
            })
            self.fpn = FPN()
            self.head = DetectionHead()

        def forward(self, x):
            # x: [B, N, H, W]
            b, n, h, w = x.shape
            x2d = x.reshape(b * n, 1, h, w)
            per_slice = self.encoder(x2d)
            fused = {}
            for name, feat in per_slice.items():
                _, c, fh, fw = feat.shape
                feat = feat.reshape(b, n, c, fh, fw)
                fused[name] = self.fusions[name](feat)
            pyramid = self.fpn(fused)
            return self.head(pyramid)
elif TORCH_AVAILABLE:
    print("Torch is available but torchvision is not. Install torchvision to run Option 0D.")
else:
    print("Torch unavailable. Model skipped.")

## 5. Target assignment, losses, decoding and metrics

In [ ]:
if TORCH_AVAILABLE:
    def choose_level(box: torch.Tensor) -> int:
        w = float((box[2] - box[0]).clamp(min=1).detach().cpu())
        h = float((box[3] - box[1]).clamp(min=1).detach().cpu())
        size = max(w, h)
        for i, (lo, hi) in enumerate(CFG.level_box_ranges):
            if size >= lo and size <= hi:
                return i
        return len(CFG.level_names) - 1


    def build_targets(outputs, boxes, has_box):
        targets = {}
        device = boxes.device
        for level_name, stride in zip(CFG.level_names, CFG.strides):
            cls_logits = outputs[level_name]["cls"]
            b, _, h, w = cls_logits.shape
            targets[level_name] = {
                "cls": torch.zeros((b, 1, h, w), dtype=torch.float32, device=device),
                "box": torch.zeros((b, 4, h, w), dtype=torch.float32, device=device),
                "pos": torch.zeros((b, 1, h, w), dtype=torch.bool, device=device),
            }

        for bi in range(boxes.shape[0]):
            if has_box[bi].item() < 0.5:
                continue
            box = boxes[bi]
            level_idx = choose_level(box)
            level_name = CFG.level_names[level_idx]
            stride = CFG.strides[level_idx]
            target = targets[level_name]
            _, _, h, w = target["cls"].shape
            x1, y1, x2, y2 = box
            cx = 0.5 * (x1 + x2)
            cy = 0.5 * (y1 + y2)
            gx = int(torch.clamp(torch.floor(cx / stride), 0, w - 1).item())
            gy = int(torch.clamp(torch.floor(cy / stride), 0, h - 1).item())
            r = int(CFG.center_radius_cells)
            for yy in range(max(0, gy - r), min(h, gy + r + 1)):
                for xx in range(max(0, gx - r), min(w, gx + r + 1)):
                    px = (xx + 0.5) * stride
                    py = (yy + 0.5) * stride
                    if px < x1 or px > x2 or py < y1 or py > y2:
                        continue
                    l = px - x1
                    t = py - y1
                    rr = x2 - px
                    bb = y2 - py
                    if min(l, t, rr, bb) <= 0:
                        continue
                    target["cls"][bi, 0, yy, xx] = 1.0
                    target["box"][bi, :, yy, xx] = torch.stack([l, t, rr, bb])
                    target["pos"][bi, 0, yy, xx] = True
        return targets


    def sigmoid_focal_loss(logits, targets, alpha=CFG.focal_alpha, gamma=CFG.focal_gamma, eps=1e-6):
        prob = torch.sigmoid(logits)
        ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p_t = prob * targets + (1 - prob) * (1 - targets)
        alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
        return alpha_t * ((1 - p_t).clamp(min=eps) ** gamma) * ce


    def detection_loss(outputs, boxes, has_box):
        targets = build_targets(outputs, boxes, has_box)
        total_cls = 0.0
        total_box = 0.0
        total_pos = 0
        for level_name, stride in zip(CFG.level_names, CFG.strides):
            cls_logits = outputs[level_name]["cls"]
            box_raw = outputs[level_name]["box"]
            t = targets[level_name]
            pos = t["pos"]
            num_pos = int(pos.sum().detach().cpu())
            total_pos += num_pos
            total_cls = total_cls + sigmoid_focal_loss(cls_logits, t["cls"]).sum()
            if num_pos > 0:
                pred_dist = F.softplus(box_raw) * float(stride)
                total_box = total_box + F.smooth_l1_loss(pred_dist[pos.expand_as(pred_dist)], t["box"][pos.expand_as(t["box"])], reduction="sum", beta=1.0)

        normalizer = max(total_pos, 1)
        cls_loss = total_cls / normalizer
        box_loss = total_box / max(total_pos * 4, 1)
        loss = CFG.lambda_cls * cls_loss + CFG.lambda_box * box_loss
        return loss, {"loss": float(loss.detach().cpu()), "cls": float(cls_loss.detach().cpu()), "box": float(box_loss.detach().cpu()), "num_pos": total_pos}


    def box_iou_torch(boxes1, boxes2):
        if boxes1.numel() == 0 or boxes2.numel() == 0:
            return torch.zeros((boxes1.shape[0], boxes2.shape[0]), device=boxes1.device)
        lt = torch.maximum(boxes1[:, None, :2], boxes2[None, :, :2])
        rb = torch.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])
        wh = (rb - lt).clamp(min=0)
        inter = wh[:, :, 0] * wh[:, :, 1]
        area1 = ((boxes1[:, 2] - boxes1[:, 0]).clamp(min=0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0))[:, None]
        area2 = ((boxes2[:, 2] - boxes2[:, 0]).clamp(min=0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0))[None, :]
        return inter / (area1 + area2 - inter + 1e-6)


    def simple_nms(boxes, scores, iou_threshold=0.3):
        if boxes.numel() == 0:
            return torch.empty((0,), dtype=torch.long, device=boxes.device)
        order = scores.argsort(descending=True)
        keep = []
        while order.numel() > 0:
            i = order[0]
            keep.append(i)
            if order.numel() == 1:
                break
            ious = box_iou_torch(boxes[i][None], boxes[order[1:]])[0]
            order = order[1:][ious <= iou_threshold]
        return torch.stack(keep)


    @torch.no_grad()
    def decode_outputs(outputs, score_threshold: float = CFG.score_threshold, nms_iou: float = CFG.nms_iou, max_dets: int = CFG.max_detections_per_slice):
        batch_size = next(iter(outputs.values()))["cls"].shape[0]
        decoded = []
        for bi in range(batch_size):
            all_boxes = []
            all_scores = []
            for level_name, stride in zip(CFG.level_names, CFG.strides):
                cls = torch.sigmoid(outputs[level_name]["cls"][bi, 0])
                box = F.softplus(outputs[level_name]["box"][bi]) * float(stride)
                keep = cls > score_threshold
                if keep.sum() == 0:
                    continue
                scores = cls[keep]
                ys, xs = keep.nonzero(as_tuple=True)
                if scores.numel() > 1000:
                    scores, top_idx = scores.topk(1000)
                    ys = ys[top_idx]
                    xs = xs[top_idx]
                dist = box[:, ys, xs].permute(1, 0)
                px = (xs.float() + 0.5) * float(stride)
                py = (ys.float() + 0.5) * float(stride)
                boxes = torch.stack([
                    px - dist[:, 0],
                    py - dist[:, 1],
                    px + dist[:, 2],
                    py + dist[:, 3],
                ], dim=1)
                boxes[:, 0::2] = boxes[:, 0::2].clamp(0, CFG.image_size)
                boxes[:, 1::2] = boxes[:, 1::2].clamp(0, CFG.image_size)
                valid = (boxes[:, 2] > boxes[:, 0] + 1) & (boxes[:, 3] > boxes[:, 1] + 1)
                if valid.sum() == 0:
                    continue
                all_boxes.append(boxes[valid])
                all_scores.append(scores[valid])
            if all_boxes:
                boxes = torch.cat(all_boxes, dim=0)
                scores = torch.cat(all_scores, dim=0)
                if torchvision_nms is not None:
                    keep = torchvision_nms(boxes, scores, nms_iou)
                else:
                    keep = simple_nms(boxes, scores, nms_iou)
                keep = keep[:max_dets]
                decoded.append({"boxes": boxes[keep].detach().cpu().numpy(), "scores": scores[keep].detach().cpu().numpy()})
            else:
                decoded.append({"boxes": np.zeros((0, 4), dtype=np.float32), "scores": np.zeros((0,), dtype=np.float32)})
        return decoded
else:
    print("Torch unavailable. Detection losses skipped.")

In [ ]:
def box_iou_np(box_a: np.ndarray, box_b: np.ndarray) -> float:
    xa1, ya1, xa2, ya2 = box_a.astype(float)
    xb1, yb1, xb2, yb2 = box_b.astype(float)
    ix1, iy1 = max(xa1, xb1), max(ya1, yb1)
    ix2, iy2 = min(xa2, xb2), min(ya2, yb2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, xa2 - xa1) * max(0.0, ya2 - ya1)
    area_b = max(0.0, xb2 - xb1) * max(0.0, yb2 - yb1)
    return float(inter / (area_a + area_b - inter + 1e-6))


def average_precision_from_pr(tp_flags: List[int], fp_flags: List[int], n_gt: int) -> float:
    if n_gt == 0 or len(tp_flags) == 0:
        return 0.0
    tp = np.cumsum(np.asarray(tp_flags, dtype=np.float32))
    fp = np.cumsum(np.asarray(fp_flags, dtype=np.float32))
    recall = tp / max(n_gt, 1)
    precision = tp / np.maximum(tp + fp, 1e-6)
    mrec = np.concatenate([[0.0], recall, [1.0]])
    mpre = np.concatenate([[0.0], precision, [0.0]])
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))


def summarize_detection_records(records: pd.DataFrame, score_threshold: float, iou_threshold: float = 0.30) -> Dict:
    df = records.copy()
    df["pred_present"] = df["score"] >= score_threshold
    df["hit"] = df["pred_present"] & df["gt_present"] & (df["best_iou"] >= iou_threshold)
    tp = int(df["hit"].sum())
    fp = int((df["pred_present"] & (~df["hit"])).sum())
    fn = int((df["gt_present"] & (~df["hit"])).sum())
    tn = int(((~df["gt_present"]) & (~df["pred_present"])).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-6)
    mean_iou_pos = float(df.loc[df["gt_present"], "best_iou"].mean()) if df["gt_present"].any() else 0.0
    empty_fp_rate = fp / max(int((~df["gt_present"]).sum()), 1)
    return {
        "score_threshold": float(score_threshold),
        "iou_threshold": float(iou_threshold),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1),
        "mean_iou_on_gt_positive": mean_iou_pos,
        "empty_fp_rate": float(empty_fp_rate),
    }


def ap_from_records(records: pd.DataFrame, iou_threshold: float = 0.30) -> float:
    n_gt = int(records["gt_present"].sum())
    pred_rows = records[records["score"] > 0].copy()
    if len(pred_rows) == 0:
        return 0.0
    pred_rows = pred_rows.sort_values("score", ascending=False)
    matched = set()
    tp_flags, fp_flags = [], []
    for _, row in pred_rows.iterrows():
        key = (row["scan_id"], int(row["slice_idx"]))
        if bool(row["gt_present"]) and float(row["best_iou"]) >= iou_threshold and key not in matched:
            tp_flags.append(1)
            fp_flags.append(0)
            matched.add(key)
        else:
            tp_flags.append(0)
            fp_flags.append(1)
    return average_precision_from_pr(tp_flags, fp_flags, n_gt)

## 6. Training setup

In [ ]:
if TORCH_AVAILABLE and TORCHVISION_AVAILABLE:
    def make_loader(ds, batch_size, positive_weight, shuffle=False):
        if shuffle:
            weights = np.where(ds.slice_df["positive"].to_numpy(bool), positive_weight, 1.0).astype(np.float64)
            sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
            return DataLoader(ds, batch_size=batch_size, sampler=sampler, shuffle=False, num_workers=CFG.num_workers, pin_memory=(DEVICE == "cuda"))
        return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=(DEVICE == "cuda"))


    train_ds = PheDetection25DDataset(train_rows, CFG.image_size, augment=CFG.augment_train)
    val_ds = PheDetection25DDataset(val_rows, CFG.image_size, augment=False)
    test_ds = PheDetection25DDataset(test_rows, CFG.image_size, augment=False)

    train_loader = make_loader(train_ds, CFG.batch_size, CFG.positive_slice_weight, shuffle=True)
    val_loader = make_loader(val_ds, CFG.batch_size, CFG.positive_slice_weight, shuffle=False)
    test_loader = make_loader(test_ds, CFG.batch_size, CFG.positive_slice_weight, shuffle=False)

    model = Phe25DDetectionModel().to(DEVICE)
    backbone_params = list(model.encoder.parameters())
    head_params = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": CFG.lr_backbone},
            {"params": head_params, "lr": CFG.lr_head},
        ],
        weight_decay=CFG.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)

    display(pd.DataFrame([
        {"split": "train", "cases": len(train_rows), "slices": len(train_ds), "positive_slices": int(train_ds.slice_df["positive"].sum())},
        {"split": "val", "cases": len(val_rows), "slices": len(val_ds), "positive_slices": int(val_ds.slice_df["positive"].sum())},
        {"split": "test", "cases": len(test_rows), "slices": len(test_ds), "positive_slices": int(test_ds.slice_df["positive"].sum())},
    ]))
    print("parameters:", sum(p.numel() for p in model.parameters()))
else:
    print("Training setup skipped.")

## 7. Train detection model

In [ ]:
if TORCH_AVAILABLE and TORCHVISION_AVAILABLE:
    def set_backbone_trainable(model, trainable: bool):
        for p in model.encoder.parameters():
            p.requires_grad = bool(trainable)


    def train_epoch(epoch: int):
        backbone_trainable = epoch > CFG.freeze_backbone_epochs
        set_backbone_trainable(model, backbone_trainable)
        model.train()
        if not backbone_trainable:
            model.encoder.eval()

        scaler = torch.cuda.amp.GradScaler(enabled=(CFG.amp and DEVICE == "cuda"))
        losses = []
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(tqdm(train_loader, desc=f"train {epoch}", leave=False), start=1):
            x = batch["image"].to(DEVICE, non_blocking=True)
            boxes = batch["box"].to(DEVICE, non_blocking=True)
            has_box = batch["has_box"].to(DEVICE, non_blocking=True).view(-1)
            with torch.cuda.amp.autocast(enabled=(CFG.amp and DEVICE == "cuda")):
                outputs = model(x)
                loss, parts = detection_loss(outputs, boxes, has_box)
                loss = loss / max(1, CFG.grad_accum_steps)
            scaler.scale(loss).backward()
            if step % CFG.grad_accum_steps == 0 or step == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            losses.append(parts["loss"])
        return float(np.mean(losses)) if losses else np.nan


    @torch.no_grad()
    def validate_epoch():
        model.eval()
        losses = []
        scores = []
        gt = []
        for batch in tqdm(val_loader, desc="val", leave=False):
            x = batch["image"].to(DEVICE, non_blocking=True)
            boxes = batch["box"].to(DEVICE, non_blocking=True)
            has_box = batch["has_box"].to(DEVICE, non_blocking=True).view(-1)
            outputs = model(x)
            loss, parts = detection_loss(outputs, boxes, has_box)
            losses.append(parts["loss"])
            max_scores = []
            for bi in range(x.shape[0]):
                s = 0.0
                for level in CFG.level_names:
                    s = max(s, float(torch.sigmoid(outputs[level]["cls"][bi, 0]).max().detach().cpu()))
                max_scores.append(s)
            scores.extend(max_scores)
            gt.extend(has_box.detach().cpu().numpy().astype(bool).tolist())
        scores = np.asarray(scores, dtype=np.float32)
        gt = np.asarray(gt, dtype=bool)
        pred = scores >= CFG.score_threshold
        tp = int((pred & gt).sum())
        fp = int((pred & ~gt).sum())
        fn = int((~pred & gt).sum())
        tn = int((~pred & ~gt).sum())
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-6)
        empty_fp_rate = fp / max((~gt).sum(), 1)
        score = f1 - 0.20 * empty_fp_rate
        return {
            "loss": float(np.mean(losses)) if losses else np.nan,
            "presence_precision": float(precision),
            "presence_recall": float(recall),
            "presence_f1": float(f1),
            "empty_fp_rate": float(empty_fp_rate),
            "score": float(score),
        }


    best_ckpt = CHECKPOINT_DIR / "option_0d_detection_best.pt"
    last_ckpt = CHECKPOINT_DIR / "option_0d_detection_last.pt"

    if CFG.run_training:
        epochs = CFG.debug_epochs if CFG.debug_small_run else CFG.epochs
        history = []
        best = -np.inf
        bad = 0
        for epoch in range(1, epochs + 1):
            t0 = time.time()
            tr_loss = train_epoch(epoch)
            val_m = validate_epoch()
            scheduler.step(val_m["score"] if np.isfinite(val_m["score"]) else 0.0)
            row = {
                "epoch": epoch,
                "lr_backbone": optimizer.param_groups[0]["lr"],
                "lr_head": optimizer.param_groups[1]["lr"],
                "backbone_trainable": epoch > CFG.freeze_backbone_epochs,
                "sec": time.time() - t0,
                "train_loss": tr_loss,
                **{f"val_{k}": v for k, v in val_m.items()},
            }
            history.append(row)
            print(row)
            pd.DataFrame(history).to_csv(LOG_DIR / "option_0d_training_log.csv", index=False)
            torch.save({"model": model.state_dict(), "cfg": asdict(CFG), "epoch": epoch, "history": history}, last_ckpt)

            score = val_m["score"]
            if np.isfinite(score) and score > best + CFG.min_delta:
                best = score
                bad = 0
                torch.save({"model": model.state_dict(), "cfg": asdict(CFG), "epoch": epoch, "best_metric": best, "history": history}, best_ckpt)
                print("Saved best:", best_ckpt)
            else:
                bad += 1
            if bad >= CFG.patience:
                print(f"Early stopping at epoch {epoch}; best val score={best:.4f}")
                break
    else:
        print("Training disabled.")
else:
    print("Training skipped.")

## 8. Validation threshold tuning and detection evaluation

In [ ]:
if TORCH_AVAILABLE and TORCHVISION_AVAILABLE:
    def torch_load_checkpoint(path: Path):
        try:
            return torch.load(path, map_location=DEVICE, weights_only=False)
        except TypeError:
            return torch.load(path, map_location=DEVICE)


    def load_best_model():
        ckpt = CHECKPOINT_DIR / "option_0d_detection_best.pt"
        if not ckpt.exists():
            ckpt = CHECKPOINT_DIR / "option_0d_detection_last.pt"
        if not ckpt.exists():
            return None, None
        m = Phe25DDetectionModel().to(DEVICE)
        state = torch_load_checkpoint(ckpt)
        state_dict = state["model"] if isinstance(state, dict) and "model" in state else state
        m.load_state_dict(state_dict)
        m.eval()
        return m, ckpt


    @torch.no_grad()
    def collect_detection_records(m, loader, score_threshold_for_decode: float = 0.01, desc: str = "eval"):
        m.eval()
        rows = []
        for batch in tqdm(loader, desc=desc, leave=False):
            x = batch["image"].to(DEVICE, non_blocking=True)
            outputs = m(x)
            decoded = decode_outputs(outputs, score_threshold=score_threshold_for_decode, nms_iou=CFG.nms_iou, max_dets=CFG.max_detections_per_slice)
            boxes_gt = batch["box"].numpy()
            has_gt = batch["has_box"].numpy().reshape(-1) > 0.5
            for i, det in enumerate(decoded):
                pred_boxes = det["boxes"]
                pred_scores = det["scores"]
                best_iou = 0.0
                best_score = 0.0
                best_box = np.zeros(4, dtype=np.float32)
                if len(pred_boxes):
                    best_score = float(pred_scores.max())
                    best_idx = int(pred_scores.argmax())
                    best_box = pred_boxes[best_idx].astype(np.float32)
                    if has_gt[i]:
                        best_iou = max(box_iou_np(b, boxes_gt[i]) for b in pred_boxes)
                rows.append({
                    "scan_id": batch["scan_id"][i],
                    "slice_idx": int(batch["slice_idx"][i]),
                    "gt_present": bool(has_gt[i]),
                    "gt_x1": float(boxes_gt[i, 0]),
                    "gt_y1": float(boxes_gt[i, 1]),
                    "gt_x2": float(boxes_gt[i, 2]),
                    "gt_y2": float(boxes_gt[i, 3]),
                    "score": best_score,
                    "best_iou": float(best_iou),
                    "pred_x1": float(best_box[0]),
                    "pred_y1": float(best_box[1]),
                    "pred_x2": float(best_box[2]),
                    "pred_y2": float(best_box[3]),
                    "num_pred": int(len(pred_boxes)),
                })
        return pd.DataFrame(rows)


    eval_model, used_ckpt = load_best_model()
    tuned_score_threshold = CFG.score_threshold
    val_records = pd.DataFrame()
    threshold_table = pd.DataFrame()

    if eval_model is not None:
        val_records = collect_detection_records(eval_model, val_loader, score_threshold_for_decode=0.01, desc="val detection")
        val_records.to_csv(TABLE_DIR / "option_0d_val_detection_records.csv", index=False)
        sweep = []
        for score_thr in CFG.score_grid:
            for iou_thr in CFG.eval_iou_thresholds:
                row = summarize_detection_records(val_records, score_threshold=score_thr, iou_threshold=iou_thr)
                row["ap"] = ap_from_records(val_records[val_records["score"] >= score_thr], iou_threshold=iou_thr)
                sweep.append(row)
        threshold_table = pd.DataFrame(sweep)
        threshold_table.to_csv(TABLE_DIR / "option_0d_val_threshold_sweep.csv", index=False)
        primary = threshold_table[threshold_table["iou_threshold"] == 0.30].copy()
        tuned_score_threshold = float(primary.sort_values(["f1", "empty_fp_rate", "mean_iou_on_gt_positive"], ascending=[False, True, False]).iloc[0]["score_threshold"])
        print("Loaded checkpoint:", used_ckpt)
        print("Tuned score threshold:", tuned_score_threshold)
        display(threshold_table)
    else:
        print("No checkpoint found for evaluation.")

    with open(LOG_DIR / "option_0d_threshold.json", "w", encoding="utf-8") as f:
        json.dump(
            {
                "score_threshold": tuned_score_threshold,
                "default_score_threshold": CFG.score_threshold,
                "checkpoint": str(used_ckpt) if used_ckpt else None,
                "selection_metric": "validation F1 at IoU 0.30; tie-breaker lower empty FP rate then higher mean IoU",
                "nms_iou": CFG.nms_iou,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )
else:
    print("Validation threshold tuning skipped.")

## 9. Final test evaluation

In [ ]:
if TORCH_AVAILABLE and TORCHVISION_AVAILABLE and CFG.run_final_eval:
    if "eval_model" not in globals() or eval_model is None:
        eval_model, used_ckpt = load_best_model()
    if "tuned_score_threshold" not in globals():
        tuned_score_threshold = CFG.score_threshold

    if eval_model is not None:
        test_records = collect_detection_records(eval_model, test_loader, score_threshold_for_decode=0.01, desc="test detection")
        test_records.to_csv(TABLE_DIR / "option_0d_test_detection_records.csv", index=False)
        metric_rows = []
        for iou_thr in CFG.eval_iou_thresholds:
            row = summarize_detection_records(test_records, score_threshold=tuned_score_threshold, iou_threshold=iou_thr)
            row["ap"] = ap_from_records(test_records[test_records["score"] >= tuned_score_threshold], iou_threshold=iou_thr)
            metric_rows.append(row)
        test_metrics = pd.DataFrame(metric_rows)
        test_metrics.to_csv(TABLE_DIR / "option_0d_test_detection_metrics.csv", index=False)
        display(test_metrics)
    else:
        print("No model checkpoint found.")
else:
    print("Final eval skipped.")

## 10. Qualitative detection boxes and protocol

In [ ]:
if "test_records" in globals() and len(test_records):
    show_df = pd.concat([
        test_records[test_records["gt_present"]].sort_values("best_iou", ascending=False).head(4),
        test_records[test_records["gt_present"]].sort_values("best_iou", ascending=True).head(4),
        test_records[~test_records["gt_present"]].sort_values("score", ascending=False).head(4),
    ], ignore_index=True).drop_duplicates(["scan_id", "slice_idx"]).head(8)

    fig, axes = plt.subplots(len(show_df), 1, figsize=(6, 5 * max(len(show_df), 1)))
    if len(show_df) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, show_df.iterrows()):
        case = test_rows[test_rows["scan_id"].astype(str) == str(row["scan_id"])].iloc[0]
        image, _, _ = load_nifti(Path(case["img_path"]))
        stack = make_25d_stack(image, int(row["slice_idx"]), image_size=CFG.image_size)
        ax.imshow(stack[CFG.stack_size // 2], cmap="gray")
        if bool(row["gt_present"]):
            gx1, gy1, gx2, gy2 = row["gt_x1"], row["gt_y1"], row["gt_x2"], row["gt_y2"]
            ax.add_patch(plt.Rectangle((gx1, gy1), gx2 - gx1, gy2 - gy1, fill=False, edgecolor="lime", linewidth=2, label="GT"))
        if row["score"] >= tuned_score_threshold:
            px1, py1, px2, py2 = row["pred_x1"], row["pred_y1"], row["pred_x2"], row["pred_y2"]
            ax.add_patch(plt.Rectangle((px1, py1), px2 - px1, py2 - py1, fill=False, edgecolor="red", linewidth=2, label="Pred"))
        ax.set_title(f"{row['scan_id']} z={int(row['slice_idx'])} score={row['score']:.3f} IoU={row['best_iou']:.3f}")
        ax.axis("off")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "option_0d_qualitative_detection_boxes.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("No test records available. Run final evaluation first.")


protocol = {
    "option": "Option 0D - PHE-only 2.5D detection",
    "task": "slice-level PHE bbox detection, not segmentation",
    "data": "PHE-SICH-CT-IDS only",
    "label_source": "PHE masks converted to union bounding box on center slice",
    "no_source_ich": True,
    "no_seg_cq500": True,
    "no_instance": True,
    "split": "Reuse Option 1 patient-level 4:4:2 split",
    "architecture": {
        "input": f"2.5D stack with offsets {list(CFG.slice_offsets)}",
        "backbone": "2D ResNet18 applied per slice",
        "fusion": "depthwise Conv3D over stacked feature maps at each scale",
        "neck": "FPN",
        "head": "FCOS-like anchor-free bbox + objectness head",
    },
    "metrics": [
        "Precision/Recall/F1 at IoU 0.30 and 0.50",
        "Specificity",
        "empty false positive rate",
        "AP at IoU thresholds",
        "mean best IoU on GT-positive slices",
    ],
}
with open(LOG_DIR / "option_0d_protocol.json", "w", encoding="utf-8") as f:
    json.dump(protocol, f, ensure_ascii=False, indent=2, default=str)
print("Saved protocol:", LOG_DIR / "option_0d_protocol.json")